# 1. Quickstart

Load a model from CSV, step it, change an input, read the snapshot and the trace.

The model is in `examples/quickstart/`: one state machine (`Lamp`: OFF/ON) driven by a boolean input `sw`, running at 10 Hz.

In [1]:
import sys, pathlib
try:
    import statesim
except ImportError:                      # not pip-installed: use the repo's src/ folder
    sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))
    import statesim
from statesim import load_model, Engine, ListTraceSink, FileTraceWriter, ModelError, EvaluationError
print("statesim imported")

statesim imported


## Load the model and create the engine
`load_model` takes the **master CSV**. `ListTraceSink` collects the log in memory (use `FileTraceWriter("run.log")` for a file).

In [2]:
model = load_model("../examples/quickstart/master.csv")
trace = ListTraceSink()
eng = Engine(model, trace=trace)
print("tick rate (Hz):", model.tick_hz)
print(eng.snapshot())

tick rate (Hz): 10
{'tick': 0, 't': 0.0, 'states': {'Lamp': 'OFF'}, 'outputs': {'off_req': False, 'on_req': False}, 'inputs': {'sw': False}}


## Step the simulation
The first `step()` is tick 1 at `t = 0 ms`. Time is absolute simulation time in **milliseconds**.

In [3]:
eng.step()
print(eng.snapshot())

{'tick': 1, 't': 0.0, 'states': {'Lamp': 'OFF'}, 'outputs': {'off_req': True, 'on_req': False}, 'inputs': {'sw': False}}


## Change an input, run several ticks
`set_input` writes an external input; the change is seen at the next evaluation. `run(n)` = `n` calls to `step()`.

In [4]:
eng.set_input("sw", True)
eng.run(2)
print(eng.snapshot())

eng.set_input("sw", False)
eng.run(2)
print(eng.snapshot())

{'tick': 3, 't': 200.0, 'states': {'Lamp': 'ON'}, 'outputs': {'off_req': False, 'on_req': True}, 'inputs': {'sw': True}}
{'tick': 5, 't': 400.0, 'states': {'Lamp': 'OFF'}, 'outputs': {'off_req': True, 'on_req': False}, 'inputs': {'sw': False}}


## Read the trace
One line per input change, result change or state transition: `tick | t (ms) | component | action | values involved`.

In [5]:
for line in trace.lines():
    print(line)

tick=000001 t=0.000 | Lamp | off_req False->True | sw=False
tick=000001 t=0.000 | INPUT | set sw False->True | sw=True
tick=000002 t=100.000 | Lamp | on_req False->True | sw=True
tick=000002 t=100.000 | Lamp | off_req True->False | sw=True
tick=000002 t=100.000 | Lamp | T1 OFF->ON "Turn on" | sw=True
tick=000003 t=200.000 | INPUT | set sw True->False | sw=False
tick=000004 t=300.000 | Lamp | on_req True->False | sw=False
tick=000004 t=300.000 | Lamp | off_req False->True | sw=False
tick=000004 t=300.000 | Lamp | T2 ON->OFF "Turn off" | sw=False


## Snapshot layout
`states` = every state machine's current state (strings), `outputs` = every result variable, `inputs` = external inputs. The snapshot is a copy: changing it does not affect the engine.

In [6]:
snap = eng.snapshot()
for key in ("tick", "t", "states", "outputs", "inputs"):
    print(f"{key:8}", snap[key])
print("engine.tick =", eng.tick, "  engine.t =", eng.t, "ms")

tick     5
t        400.0
states   {'Lamp': 'OFF'}
outputs  {'off_req': True, 'on_req': False}
inputs   {'sw': False}
engine.tick = 5   engine.t = 400 ms
